# Асинхронное программирование
**Ассинхронное программирование** - концепция программирования, при применении которой запуск длительных операций происходит без ожидания их завершения и не блокирует дальнейшее выполнение программы. Проще говоря, операции запускаются в фоновом режиме, однако стоит учитывать, что в таком случае результат операции будет доступен лишь какое-то время спустя.

In [125]:
import asyncio
import time
import requests
import aiohttp


In [126]:
async def task(name, time):
    print(f"{name}: начал")
    await asyncio.sleep(time)  # НЕ БЛОКИРУЕТ главный поток (выполняются другие задачи)
    print(f"{name}: закончил")

async def main():
    # Запуск задач 
    task1 = asyncio.create_task(task("Задача 1", 3))
    task2 = asyncio.create_task(task("Задача 2", 2))
    task3 = asyncio.create_task(task("Задача 3", 1))
    print("Все задачи запущены")
    # Ожидание их завершения 
    await asyncio.gather(task1, task2, task3)
    print("Все задачи завершены")

# asyncio.run(main())
await main()

Все задачи запущены
Задача 1: начал
Задача 2: начал
Задача 3: начал
Задача 3: закончил
Задача 2: закончил
Задача 1: закончил
Все задачи завершены


В Jupyter вместо ***asyncio.run(main())*** используем ***await main()*** в связи с тонкостями работы среды Jupyter, которая уже имеет запущенный цикл событий.:

Важно понимать, что асинхронность - это не одновременное выполнение нескольких процессов, а лишь переключение между ними с помощью sleep. 

К примеру: мы отправляем запрос на сайт и переключаемся на другую задачу, когда возвращаемся к первой, запрос на сайт уже прошёл, и мы получили результат, без ожидания ответа от сайта.

Этим асинхронность отличается от многопоточности:
***
Многопоточность

Поток 1: [====работа====][---ожидание---][====работа====]

Поток 2: [====работа====][---ожидание---][====работа====]

Поток 3: [====работа====][---ожидание---][====работа====]
***
Асинхронность

Поток 1: [работа1][---ожидание1---][работа2][---ожидание2---][работа3]

ТОЛЬКО ОДНА задача в момент времени, быстро переключается

# 🌐 Пример использования и сравнение

In [127]:
# БЕЗ АСИНХРОННОСТИ (синхронно)
def without_async():
    print("\n БЕЗ асинхронности:")
    start = time.time()
    
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/1"
    ]
    
    for url in urls:
        response = requests.get(url)  # БЛОКИРУЕТСЯ на 1 секунду
        print(f"  Получен ответ от {url}")
    
    print(f"  Время: {time.time() - start:.2f} секунд")

# С АСИНХРОННОСТЬЮ
async def with_async():
    print("\n С асинхронностью:")
    start = time.time()
    
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/1"
    ]
    
    connector = aiohttp.TCPConnector(ssl=False)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [session.get(url) for url in urls]
        responses = await asyncio.gather(*tasks)  # ВСЕ ПАРАЛЛЕЛЬНО!
        
        for resp in responses:
            await resp.text()
            print(f"  Получен ответ")
    
    print(f"  Время: {time.time() - start:.2f} секунд")

# Запускаем сравнение
without_async()
await with_async()


 БЕЗ асинхронности:
  Получен ответ от https://httpbin.org/delay/1
  Получен ответ от https://httpbin.org/delay/1
  Получен ответ от https://httpbin.org/delay/1
  Время: 5.06 секунд

 С асинхронностью:
  Получен ответ
  Получен ответ
  Получен ответ
  Время: 2.60 секунд


# 📦 Основные асинхронные библиотеки Python

## 🎯 Быстрый выбор

| Если вам нужно... | Используйте... |
|------------------|----------------|
| HTTP запросы | `httpx` (простой) или `aiohttp` (мощный) |
| Веб-сервер API | `fastapi` |
| База данных PostgreSQL | `asyncpg` |
| База данных SQLite | `aiosqlite` |
| MongoDB | `motor` |
| Redis | `redis-py` |
| Читать/писать файлы | `aiofiles` |
| Telegram бот | `aiogram` |
| Ускорить asyncio | `uvloop` |
| Сложную конкурентность | `anyio` |
| Тестирование | `pytest-asyncio` |

## 🛠️ И другие:

| Категория | Библиотека | Назначение |
|-----------|------------|-------------|
| **HTTP клиент/сервер** | `aiohttp` | Самая популярная библиотека для HTTP-запросов и WebSocket (клиент + сервер) |
| **HTTP клиент** | `httpx` | Современный HTTP клиент с API как у `requests`. Поддерживает HTTP/2, работает синхронно и асинхронно |
| **HTTP сервер** | `fastapi` | Высокопроизводительный веб-фреймворк для создания API (на базе Starlette) |
| **HTTP сервер** | `starlette` | Легковесный ASGI фреймворк для веб-приложений |
| **HTTP сервер** | `sanic` | Веб-фреймворк, вдохновленный Flask, с поддержкой асинхронных обработчиков |
| **WebSocket** | `websockets` | Библиотека для работы с WebSocket (низкоуровневая, надежная) |
| **PostgreSQL** | `asyncpg` | Самый быстрый асинхронный драйвер для PostgreSQL |
| **MySQL/MariaDB** | `aiomysql` | Асинхронный драйвер для MySQL (на основе PyMySQL) |
| **MySQL/MariaDB** | `asyncmy` | Более быстрая альтернатива aiomysql (написана на Cython) |
| **SQLite** | `aiosqlite` | Асинхронная обертка для SQLite |
| **MongoDB** | `motor` | Официальный асинхронный драйвер MongoDB |
| **Redis** | `redis-py` | Асинхронный клиент Redis (начиная с версии 4.2.0) |
| **Redis** | `aioredis` | Популярный асинхронный клиент Redis (часть redis-py) |
| **Файлы** | `aiofiles` | Асинхронная работа с файлами (чтение/запись) |
| **Файлы** | `aiofiles-x` | Ускоренная версия aiofiles на C++23 |
| **Процессы** | `asyncio.subprocess` | Встроенный модуль для асинхронного запуска внешних процессов |
| **Очереди** | `aio-pika` | Асинхронный клиент для RabbitMQ |
| **Очереди** | `aiokafka` | Асинхронный клиент для Apache Kafka |
| **S3/Облака** | `aioboto3` | Асинхронный клиент для AWS S3 и других сервисов AWS |
| **Альтернативы asyncio** | `anyio` | Высокоуровневый фреймворк для конкурентности (работает на asyncio и trio) |
| **Альтернативы asyncio** | `trio` | Альтернативный движок с более безопасной моделью конкурентности |
| **Ускорение** | `uvloop` | Замена стандартного event loop (быстрее в 2-4 раза) |
| **Тестирование** | `pytest-asyncio` | Поддержка асинхронных тестов в pytest |
| **Брокеры задач** | `celery` | Поддержка асинхронных задач (с async/await начиная с 5.0) |
| **Брокеры задач** | `arq` | Легковесный асинхронный брокер задач на Redis |
| **gRPC** | `grpcio` | Асинхронная поддержка gRPC (начиная с версии 1.32) |
| **Telegram боты** | `aiogram` | Популярный асинхронный фреймворк для Telegram Bot API |
| **Telegram боты** | `python-telegram-bot` | Поддержка асинхронности (начиная с версии 20.0) |
| **Discord боты** | `discord.py` | Асинхронная библиотека для Discord API |
| **SSH** | `asyncssh` | Асинхронный клиент и сервер SSH |
| **SMTP** | `aiosmtplib` | Асинхронный клиент для отправки email через SMTP |
| **Web scraping** | `aiohttp` + `beautifulsoup4` | Асинхронный парсинг сайтов |
| **Web scraping** | `parsel` | Асинхронная версия парсера (обычно с aiohttp) |

